Input Validation


In [ ]:
customerEmail = """ My name is rahul bisht and i work at openai and my email is rahul@gmail.com and phone number is 26361872386"""
print(customerEmail)

**Definining the blueprint**


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class CustomerEmail(BaseModel):
    name: str = Field(..., description="The name of the customer")
    email: str = Field(..., description="The email of the customer")
    phone: str = Field(..., description="The phone number of the customer")
    category: Literal["primium", "normal"] = Field(
        ..., description="The category of the customer"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        ..., description="The urgency of the customer"
    )

In [ ]:
correctEmail = CustomerEmail(
    name="rahul bisht",
    email="rahul@gmail.com",
    phone="26361872386",
    category="primium",
    urgency="high",
)

**Structured Data inside the GuardRails AI**


In [ ]:
from guardrails import Guard

guard = Guard.for_pydantic(CustomerEmail)
data = "My name is rahul bisht and i work at openai and my email is rahul@gmail.com and phone number is 26361872386"
# outcome = guard.validate(correctEmail.model_dump_json())
outcome = guard.validate(data)
print(outcome)

**Asking the Model to generate the response**
### guard(...) calls the LLM, then validates the response against your CustomerEmail Pydantic schema.
### Returns a ValidationOutcome with raw_llm_output, validated_output, and validation_passed.
result = guard(
    # Which LLM to call. Guardrails routes this through LiteLLM (needs OPENAI_API_KEY for OpenAI models).
    model="gpt-4o-mini",

    # If True, prints detailed logs during the call (LLM request, validation steps, re-asks).
    verbose=True,

    # How many times Guardrails can re-prompt the LLM if validation fails (e.g. invalid JSON or missing fields).
    # 0 = no retries; 2 = up to 2 extra attempts after the first response.
    num_reasks=2,

    # Chat history sent to the model. Same format as OpenAI's messages API.
    messages=[
        {
            # "user" = the human prompt. Other roles: "system", "assistant".
            "role": "user",

            # The actual instruction + your input text. The model reads this to extract structured data.
            "content": f"""
            Extract customer details from this text and return valid JSON.
            Text:
            {customerEmail}
            Infer category as "primium" or "normal" and urgency as low/medium/high if not stated.
            """,
        },
    ],

    ### Max tokens in the model's response. Increase if your JSON output might be long.
    max_tokens=1000,

    ### Randomness: 0 = deterministic, 1 = more creative. 0.5 is a moderate balance.
    temperature=0.5,

    ### Nucleus sampling: only consider tokens whose cumulative probability reaches this value.
    # 1 = consider all tokens (default). Lower = more focused output.
    top_p=1,

    ### Penalize tokens that appear often in the text so far. Reduces repetition. 0 = no penalty.
    frequency_penalty=0,

    ###Penalize tokens that have appeared at all. Encourages new topics. 0 = no penalty.
    presence_penalty=0,

    ### Sequences that stop generation when encountered. None = let the model finish naturally.
    stop=None,
)


In [ ]:
result = guard(
    model="gpt-4o-mini",
    verbose=True,
    num_reasks=2,
    messages=[
        {
            "role": "user",
            "content": f"""
            Extract customer details from this text and return valid JSON.
            Text:
            {customerEmail}
            Infer category as "primium" or "normal" and urgency as low/medium/high if not stated.
            """,
        },
    ],
    max_tokens=1000,
    temperature=0.5,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None,
)

print(result)